In [4]:
import os
import sys
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import pandas as pd

print("Python:", sys.version)
print("Open3D:", o3d.__version__)
print("Numpy:", np.__version__)

Python: 3.11.14 | packaged by conda-forge | (main, Oct 13 2025, 14:29:50) [Clang 19.1.7 ]
Open3D: 0.19.0
Numpy: 2.4.6


In [5]:
PCD_FOLDER_LOCATION="../../data/raw/lidar_with_intensity_and_clusters/"
PCD_FILE_NAME="frame_00006.pcd"
PCD_FILE=PCD_FOLDER_LOCATION+PCD_FILE_NAME
if not os.path.exists(PCD_FILE):
    print(f"Error: File {PCD_FILE} not found.")
else:
    # Use the Tensor API to read the point cloud, which handles custom fields like 'intensity'
    t_pcd = o3d.t.io.read_point_cloud(PCD_FILE)
    print("Loaded point cloud tensor:")
    print(t_pcd)
    # Check if 'intensity' is available
    if 'intensity' in t_pcd.point:
        print("\nIntensity field found in the point cloud!")
    else:
        print("\nWarning: No intensity field found in the PCD file. Check the file format.")


Loaded point cloud tensor:
PointCloud on CPU:0 [227351 points (Float32)].
Attributes: intensity (dtype = Float32, shape = {227351, 1}), actor_id (dtype = Float32, shape = {227351, 1}), class_id (dtype = Float32, shape = {227351, 1}), material_id (dtype = Float32, shape = {227351, 1}).

Intensity field found in the point cloud!


### 🧹 LiDAR Point Cloud Cleaning, Extraction & Merging

This section extracts spatial coordinates and intensity values from the Open3D Tensor point cloud (`t_pcd`), filters out corrupted sensor returns (`NaN` and `Inf`), and combines spatial and reflectivity data into a single unified array.

#### 🛠️ Processing Logic:
1. **Extract Positions**: Converts spatial coordinates from `t_pcd.point.positions` into a 2D NumPy array.
2. **Filter Non-Finite Values**: Generates a boolean mask (`valid_mask`) to remove points containing `NaN` or `Inf` across $(X, Y, Z)$ coordinates.
3. **Extract & Align Intensity**: Pulls intensity values from `t_pcd.point.intensity` (or generates zeros if unavailable), applies `valid_mask` to maintain index alignment, and flattens it to a 1D array.
4. **Combine Datasets**: Stacks the clean 3D positions $(X, Y, Z)$ and 1D intensity values $(I)$ horizontally into a single $(M, 4)$ array.

#### 📊 Output Variables Created:
* **`points`**: Unfiltered 2D array `(N, 3)` of raw $(X, Y, Z)$ coordinates.
* **`valid_mask`**: 1D boolean array `(N,)` indicating valid points (`True`).
* **`valid_points`**: Cleaned 2D array `(M, 3)` containing only finite $(X, Y, Z)$ coordinates ($M \le N$).
* **`intensities`**: Synchronized 1D array `(M,)` of laser intensity values.
* **`points_with_intensity`**: Unified 2D array `(M, 4)` containing stacked $[X, Y, Z, I]$ data for each valid point.

In [6]:
points = t_pcd.point.positions.numpy()
valid_mask = np.isfinite(points).all(axis=1)

# Apply mask to coordinates
valid_points = points[valid_mask]
print(f"Total points before filtering: {len(points)}")
print(f"Valid points after filtering: {len(valid_points)}")

# Extract intensity and apply mask
if 'intensity' in t_pcd.point:
    intensities = t_pcd.point.intensity.numpy()[valid_mask].flatten()
    print(f"Intensity range: Min={intensities.min():.2f}, Max={intensities.max():.2f}")
else:
    intensities = np.zeros(len(valid_points))
# Combine XYZ positions and intensity into a single (M, 4) array
points_with_intensity = np.hstack((valid_points, intensities.reshape(-1, 1)))

Total points before filtering: 227351
Valid points after filtering: 52362
Intensity range: Min=1.00, Max=255.00


### 🌐 Polar Coordinate Conversion & Feature Augmentation

This section calculates 2D polar coordinates—range distance ($r$) and azimuth angle ($\theta$)—from the horizontal 3D coordinates $(X, Y)$ and appends them as new features to the LiDAR point cloud dataset.

#### 🛠️ Mathematical & Processing Logic:
1. **Extract 2D Coordinates**: Slices column `0` ($X$) and column `1` ($Y$) from the `points_with_intensity` array.
2. **Calculate Range ($r$)**: Computes horizontal distance from the LiDAR sensor origin using the Euclidean formula $r = \sqrt{X^2 + Y^2}$.
3. **Calculate Azimuth ($\theta$)**: Computes the horizontal angle in radians using `np.arctan2(Y, X)`, preserving quadrant signs ($-\pi$ to $+\pi$).
4. **Reshape & Stack**: Reshapes $r$ and $\theta$ into 2D column vectors `(M, 1)` and stacks them horizontally onto `points_with_intensity`.

#### 📊 Output Variables Created:
* **`x` / `y`**: 1D arrays `(M,)` containing extracted horizontal spatial coordinates.
* **`r` / `r_col`**: 1D `(M,)` and 2D column vector `(M, 1)` representing radial distance (in meters).
* **`theta` / `theta_col`**: 1D `(M,)` and 2D column vector `(M, 1)` representing horizontal angle (in radians).
* **`points_XYZIrtheta`**: Augmented 2D array `(M, 6)` containing stacked $[X, Y, Z, I, r, \theta]$ attributes per point.

In [7]:
x = points_with_intensity[:, 0]
y = points_with_intensity[:, 1]
r = np.sqrt(x**2 + y**2)
theta = np.arctan2(y, x)
r_col = r.reshape(-1, 1)
theta_col = theta.reshape(-1, 1)
points_XYZIrtheta = np.hstack((points_with_intensity, r_col, theta_col))
print("New data shape:", points_XYZIrtheta.shape)
print(points_XYZIrtheta[:5])

New data shape: (52362, 6)
[[-1.90573845e+01 -1.94020441e-03 -1.59756005e+00  2.10000000e+01
   1.90573845e+01 -3.14149094e+00]
 [-1.75806713e+01  9.70237772e-04 -1.59677327e+00  2.30000000e+01
   1.75806713e+01  3.14153743e+00]
 [-1.63286304e+01  5.67246461e-03 -1.59762537e+00  2.50000000e+01
   1.63286324e+01  3.14124537e+00]
 [-1.52225885e+01 -4.24072519e-03 -1.59648776e+00  2.70000000e+01
   1.52225895e+01 -3.14131403e+00]
 [-1.42626047e+01  5.38204890e-03 -1.60177398e+00  2.80000000e+01
   1.42626057e+01  3.14121532e+00]]


### Adaptive Spatial Sampling/Aggregation
**Algorithm Explanation:**
1. **Filtering**: First, we filter the input array to only include points within 100m (`r <= 100.0`).
2. **Global XY Grid Calculation**: For each distance band (0-10m, 10-25m, etc.), we compute the grid cell indices for the points using the formula `floor(x / cell_size)` and `floor(y / cell_size)`. Because we divide the global `x` and `y` coordinates directly, the grid is perfectly aligned in the same global XY coordinate system without any independent shifts per band.
3. **Grouping & Aggregation**: We use `pandas.DataFrame.groupby` to group points by their grid cell indices (`x_idx`, `y_idx`). 
4. **Representative Points**: Within each cell, we compute the centroid (mean) of `x, y, z, and intensity`. We also compute the count, min, max, mean, and std of the `z` values.
5. **Recalculation**: Finally, `r` and `theta` are recalculated using the new representative `x` and `y` coordinates.
6. **Output**: The function returns a single concatenated pandas DataFrame containing the aggregated points across all bands, without modifying the original array.

In [8]:

def adaptive_sample(points_array):
    """
    Adaptive spatial sampling of LiDAR points based on distance (r).
    Input points_array: [x, y, z, intensity, r, theta]
    """
    # 1. Define the distance bands and their corresponding cell sizes
    bands = [
        (0, 50, 0.05),
        (50, 85, 0.25),
        (85, 100, 0.50)
    ]
    
    # Create a DataFrame for easier grouping without overwriting the original array
    # Columns correspond to [x, y, z, intensity, r, theta]
    df = pd.DataFrame(points_array, columns=['x', 'y', 'z', 'intensity', 'r', 'theta'])
    
    # Filter out points beyond 100 m
    df_filtered = df[df['r'] <= 100.0].copy()
    
    original_valid_points = len(df_filtered)
    aggregated_records = []
    band_counts = {}
    
    # Process each distance band
    for r_min, r_max, cell_size in bands:
        # Select points in the current distance band
        # Use > for min and <= for max to avoid overlapping boundaries
        if r_min == 0:
            mask = (df_filtered['r'] >= r_min) & (df_filtered['r'] <= r_max)
        else:
            mask = (df_filtered['r'] > r_min) & (df_filtered['r'] <= r_max)
            
        df_band = df_filtered[mask].copy()
        
        if df_band.empty:
            band_counts[f"{r_min}-{r_max}m"] = 0
            continue
            
        # Calculate global XY grid cell indices
        # By dividing by cell_size and using floor, we maintain a consistent global grid
        df_band['x_idx'] = np.floor(df_band['x'] / cell_size).astype(int)
        df_band['y_idx'] = np.floor(df_band['y'] / cell_size).astype(int)
        
        # Group by cell indices and aggregate
        grouped = df_band.groupby(['x_idx', 'y_idx'])
        
        # Compute representative point (centroid/mean) and z-statistics
        agg_df = grouped.agg(
            representative_x=('x', 'mean'),
            representative_y=('y', 'mean'),
            representative_z=('z', 'mean'),
            representative_intensity=('intensity', 'mean'),
            point_count=('z', 'count'),
            z_min=('z', 'min'),
            z_max=('z', 'max'),
            z_mean=('z', 'mean'),
            z_std=('z', 'std') # std is NaN for counts of 1
        ).reset_index()
        
        # Fill NaN standard deviations with 0.0 (happens when there is only 1 point in a cell)
        agg_df['z_std'] = agg_df['z_std'].fillna(0.0)
        
        # Recalculate r and theta for the representative points
        agg_df['r'] = np.sqrt(agg_df['representative_x']**2 + agg_df['representative_y']**2)
        agg_df['theta'] = np.arctan2(agg_df['representative_y'], agg_df['representative_x'])
        
        # Add identification info
        agg_df['distance_band'] = f"{r_min}-{r_max}m"
        agg_df['cell_size'] = cell_size
        
        aggregated_records.append(agg_df)
        band_counts[f"{r_min}-{r_max}m"] = len(agg_df)
        
    # Combine all distance bands
    if aggregated_records:
        final_df = pd.concat(aggregated_records, ignore_index=True)
    else:
        final_df = pd.DataFrame()
        
    # Print basic comparison statistics
    final_count = len(final_df)
    reduction = ((original_valid_points - final_count) / original_valid_points * 100) if original_valid_points > 0 else 0
    
    print("-" * 50)
    print("Adaptive Spatial Sampling Statistics")
    print("-" * 50)
    print(f"Original number of valid points (<=100m): {original_valid_points}")
    print(f"Number of points after aggregation:       {final_count}")
    print(f"Percentage reduction:                     {reduction:.2f}%")
    print("\nNumber of occupied cells in each distance band:")
    for band, count in band_counts.items():
        print(f"  {band}: {count} cells")
    print("-" * 50)
        
    return final_df

# Example execution:
# Assuming points_XYZIrtheta was defined in the previous cell
if 'points_XYZIrtheta' in locals() or 'points_XYZIrtheta' in globals():
    aggregated_data = adaptive_sample(points_XYZIrtheta)
    display(aggregated_data.head())
else:
    print("points_XYZIrtheta is not defined yet. Run the previous cells first.")


--------------------------------------------------
Adaptive Spatial Sampling Statistics
--------------------------------------------------
Original number of valid points (<=100m): 52362
Number of points after aggregation:       8685
Percentage reduction:                     83.41%

Number of occupied cells in each distance band:
  0-50m: 8685 cells
  50-85m: 0 cells
  85-100m: 0 cells
--------------------------------------------------


,x_idx,y_idx,representative_x,representative_y,representative_z,representative_intensity,point_count,z_min,z_max,z_mean,z_std,r,theta,distance_band,cell_size
0,-382,-11,-19.053368,-0.532506,-1.597454,21.0,1,-1.597454,-1.597454,-1.597454,0.0,19.060808,-3.113652,0-50m,0.05
1,-382,-9,-19.053183,-0.422271,-1.602630,21.0,1,-1.602630,-1.602630,-1.602630,0.0,19.057861,-3.119434,0-50m,0.05
2,-382,-6,-19.050480,-0.266851,-1.600799,21.0,1,-1.600799,-1.600799,-1.600799,0.0,19.052349,-3.127586,0-50m,0.05
3,-382,-5,-19.056562,-0.213897,-1.602931,21.0,1,-1.602931,-1.602931,-1.602931,0.0,19.057762,-3.130369,0-50m,0.05
4,-382,-4,-19.050646,-0.166188,-1.604884,21.0,1,-1.604884,-1.604884,-1.604884,0.0,19.051371,-3.132869,0-50m,0.05


### 3D Visualization of Aggregated Points
The following cell takes the `aggregated_data` generated by our adaptive spatial sampling and visualizes the representative points in a 3D window using Open3D. We color the points based on their `representative_intensity`.

In [9]:
# import open3d as o3d
# import matplotlib.pyplot as plt
# import numpy as np

# # Make sure aggregated_data exists from the previous cell
# if 'aggregated_data' in locals() or 'aggregated_data' in globals():
#     # Extract XYZ coordinates from the dataframe
#     pts = aggregated_data[['representative_x', 'representative_y', 'representative_z']].values
    
#     # Extract intensity for coloring
#     intensities = aggregated_data['representative_intensity'].values
    
#     # Normalize intensity to [0, 1] to map it to colors
#     intensity_norm = (intensities - intensities.min()) / (intensities.max() - intensities.min() + 1e-6)
    
#     # Use 'turbo' colormap for clear distinction of intensity
#     cmap = plt.get_cmap('turbo')
#     colors = cmap(intensity_norm)[:, :3]
    
#     # Create Open3D point cloud object
#     pcd_vis = o3d.geometry.PointCloud()
#     pcd_vis.points = o3d.utility.Vector3dVector(pts)
#     pcd_vis.colors = o3d.utility.Vector3dVector(colors)
    
#     print(f"Visualizing {len(pts)} adaptively sampled points in a new window...")
    
#     # Open the interactive visualization window
#     o3d.visualization.draw_geometries([pcd_vis], 
#                                       window_name="Adaptive Spatial Sampling Visualization", 
#                                       width=1280, 
#                                       height=720)
# else:
#     print("Error: aggregated_data is not defined. Please run the adaptive spatial sampling cell first.")


### Polar 3D Voxelization (Comparison)
Here we implement the Polar 3D Voxelization method to compare it against the Adaptive Spatial Sampling. We group the points into `(R_bins, Theta_bins, Z_bins)` and calculate the mean XYZ and intensity for each occupied bin.

In [10]:
import numpy as np
import pandas as pd

def polar_3d_voxelization(points_array, grid_size):
    """
    Transforms [x, y, z, intensity, r, theta] into 3D Polar Voxels.
    
    Args:
        points_array: NumPy array of shape (N, 6)
        grid_size: Tuple of (R_bins, Theta_bins, Z_bins) indicating how many 
                   divisions to make in each dimension.
    """
    # 1. Extract dimensions
    z = points_array[:, 2]
    r = points_array[:, 4]
    theta = points_array[:, 5]
    
    # 2. Define the absolute limits of your Lidar sensor
    # Adjust these based on your specific dataset limits
    min_r, max_r = 0.0, 100.0
    min_theta, max_theta = -np.pi, np.pi # -180 to 180 degrees
    min_z, max_z = -3.0, 3.0             # 3 meters below to 3 meters above sensor
    
    # 3. Calculate step sizes for each dimension
    r_step = (max_r - min_r) / grid_size[0]
    theta_step = (max_theta - min_theta) / grid_size[1]
    z_step = (max_z - min_z) / grid_size[2]
    
    # 4. Digitize coordinates into integer bin indices
    r_idx = np.floor((r - min_r) / r_step).astype(np.int32)
    theta_idx = np.floor((theta - min_theta) / theta_step).astype(np.int32)
    z_idx = np.floor((z - min_z) / z_step).astype(np.int32)
    
    # 5. Clip indices to ensure they stay strictly within array bounds
    r_idx = np.clip(r_idx, 0, grid_size[0] - 1)
    theta_idx = np.clip(theta_idx, 0, grid_size[1] - 1)
    z_idx = np.clip(z_idx, 0, grid_size[2] - 1)
    
    # 6. Stack indices back with the original points
    # Shape becomes (N, 3): Each row is the [r_idx, theta_idx, z_idx] for a point
    voxel_indices = np.vstack((r_idx, theta_idx, z_idx)).T
    
    # 7. Find unique voxels and the inverse mapping
    # This automatically groups all points falling into the same 3D bin
    unique_voxels, inverse_indices, point_counts = np.unique(
        voxel_indices, axis=0, return_inverse=True, return_counts=True
    )
    
    return unique_voxels, inverse_indices, point_counts

# Example Usage:
if 'points_XYZIrtheta' in locals() or 'points_XYZIrtheta' in globals():
    # Only process points within 100m to match the adaptive spatial sampling boundary
    mask = points_XYZIrtheta[:, 4] <= 100.0
    points_to_process = points_XYZIrtheta[mask]
    
    # Define a grid of 480 distance bins, 360 angle bins, and 32 height bins.
    grid_size = (480, 360, 32)
    unique_voxels, inverse_indices, counts = polar_3d_voxelization(points_to_process, grid_size)
    
    # In order to visualize this, we need representative points for each voxel (e.g. centroids)
    df_polar = pd.DataFrame(points_to_process, columns=['x', 'y', 'z', 'intensity', 'r', 'theta'])
    df_polar['voxel_idx'] = inverse_indices
    
    polar_aggregated_data = df_polar.groupby('voxel_idx').agg(
        representative_x=('x', 'mean'),
        representative_y=('y', 'mean'),
        representative_z=('z', 'mean'),
        representative_intensity=('intensity', 'mean'),
        point_count=('z', 'count')
    ).reset_index()
    
    original_points = len(points_to_process)
    final_points = len(polar_aggregated_data)
    reduction = (original_points - final_points) / original_points * 100
    
    print("-" * 50)
    print("Polar 3D Voxelization Statistics")
    print("-" * 50)
    print(f"Original number of valid points (<=100m): {original_points}")
    print(f"Number of points after aggregation:       {final_points}")
    print(f"Percentage reduction:                     {reduction:.2f}%")
    print(f"Total non-empty bins (voxels):            {len(unique_voxels)}")
    print(f"Average points per occupied bin:          {np.mean(counts):.2f}")
    print(f"Max points in a single bin:               {np.max(counts)}")
    print("-" * 50)
    
    display(polar_aggregated_data.head())
else:
    print("points_XYZIrtheta is not defined yet.")


--------------------------------------------------
Polar 3D Voxelization Statistics
--------------------------------------------------
Original number of valid points (<=100m): 52362
Number of points after aggregation:       3506
Percentage reduction:                     93.30%
Total non-empty bins (voxels):            3506
Average points per occupied bin:          14.93
Max points in a single bin:               79
--------------------------------------------------


,voxel_idx,representative_x,representative_y,representative_z,representative_intensity,point_count
0,0,-0.580258,-0.004867,-0.200742,83.136360,44
1,1,-0.584614,-0.015492,-0.200753,82.609756,41
2,2,-0.583973,-0.026404,-0.199236,82.560974,41
3,3,-0.583042,-0.035613,-0.200685,82.571426,35
4,4,-0.576789,-0.045396,-0.200708,83.500000,38


### 3D Visualization of Polar Voxel Aggregated Points
Visualizes the representative points extracted from the Polar 3D Voxels.

In [11]:
import open3d as o3d
import matplotlib.pyplot as plt
import numpy as np

if 'polar_aggregated_data' in locals() or 'polar_aggregated_data' in globals():
    pts = polar_aggregated_data[['representative_x', 'representative_y', 'representative_z']].values
    intensities = polar_aggregated_data['representative_intensity'].values
    
    intensity_norm = (intensities - intensities.min()) / (intensities.max() - intensities.min() + 1e-6)
    cmap = plt.get_cmap('turbo')
    colors = cmap(intensity_norm)[:, :3]
    
    pcd_vis = o3d.geometry.PointCloud()
    pcd_vis.points = o3d.utility.Vector3dVector(pts)
    pcd_vis.colors = o3d.utility.Vector3dVector(colors)
    
    print(f"Visualizing {len(pts)} Polar Voxel aggregated points in a new window...")
    
    o3d.visualization.draw_geometries([pcd_vis], 
                                      window_name="Polar 3D Voxelization Visualization", 
                                      width=1280, 
                                      height=720)
else:
    print("Error: polar_aggregated_data is not defined.")


Visualizing 3506 Polar Voxel aggregated points in a new window...
